# VPD and Cattle Mortality: Correlation Analysis

This notebook examines the relationship between atmospheric vapor pressure deficit (VPD) and cattle slaughter rates in USDA Regions 4 & 6.

## Scientific Background

**Vapor Pressure Deficit (VPD)** represents the difference between the amount of moisture in the air and the maximum amount of moisture the air can hold at saturation. High VPD indicates dry air conditions that:

1. **Impair thermoregulation**: Cattle rely on evaporative cooling (sweating, panting)
2. **Increase dehydration risk**: Higher respiratory water loss
3. **Compound heat stress**: VPD amplifies temperature effects
4. **Affect feed intake**: Reduced consumption during high VPD periods

## Research Questions

1. What is the correlation between weekly VPD and cattle slaughter rates?
2. Do VPD effects show temporal lags (current vs. lagged exposure)?
3. How do VPD-mortality relationships vary by region and season?
4. Are VPD effects independent of temperature, or is there interaction?
5. What VPD thresholds correspond to elevated mortality risk?

## Data Sources

- **Climate data**: MERRA-2 afternoon VPD (12pm-6pm UTC), aggregated to weekly (1984-2025)
- **Cattle data**: USDA weekly slaughter reports, Regions 4 & 6 (1984-2027)
- **Spatial domain**: 14 states (AL, AR, AZ, FL, GA, KY, LA, MS, NC, NM, OK, SC, TN, TX)
- **Temporal resolution**: Weekly (Saturday-ending, matching USDA reporting)

## Methods

- Pearson correlation (linear relationships)
- Spearman rank correlation (monotonic relationships)
- Cross-correlation analysis (temporal lags 0-8 weeks)
- Partial correlation (controlling for temperature)
- Seasonal decomposition
- Quantile regression (response at different mortality percentiles)

In [ ]:
# Standard imports
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path
from scipy import stats, signal
from scipy.stats import pearsonr, spearmanr
from statsmodels.tsa.stattools import ccf, acf
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (16, 10)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
sns.set_palette('husl')

print("Imports complete!")

In [ ]:
# Configuration
import sys
sys.path.append(str(Path('../..')))

from config import (
    PROCESSED_WEEKLY_DIR,
    MASK_FILE,
    CATTLE_DATA_FILE,
    FIGURES_DIR,
    FOCUS_STATES,
    CATTLE_REGIONS,
    CATTLE_REGION_IDS,
    SEASONS
)

OUTPUT_DIR = FIGURES_DIR / 'vpd_mortality'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {OUTPUT_DIR}")
print(f"Focus states: {len(FOCUS_STATES)}")
print(f"Cattle Region 4: {CATTLE_REGIONS['region_4']}")
print(f"Cattle Region 6: {CATTLE_REGIONS['region_6']}")

In [ ]:
def load_weekly_data():
    """Load weekly climate and cattle data."""
    # Load weekly climate data
    ds_night = xr.open_dataset(PROCESSED_WEEKLY_DIR / 'weekly_nighttime_recovery.nc')
    ds_day = xr.open_dataset(PROCESSED_WEEKLY_DIR / 'weekly_daytime_heat.nc')
    ds_vpd = xr.open_dataset(PROCESSED_WEEKLY_DIR / 'weekly_vpd.nc')
    
    week_dates = ds_vpd['week'].values
    
    # Add convenience coordinates
    for ds in [ds_night, ds_day, ds_vpd]:
        ds.coords['week_date'] = ds['week']
        ds.coords['year'] = ('week', pd.to_datetime(week_dates).year)
        ds.coords['month'] = ('week', pd.to_datetime(week_dates).month)
    
    return ds_night, ds_day, ds_vpd, week_dates

def compute_regional_mean(data, state_ids, state_mask):
    """Compute spatial mean across multiple states."""
    combined_mask = xr.zeros_like(state_mask, dtype=bool)
    for state_id in state_ids:
        combined_mask = combined_mask | (state_mask == state_id)
    
    masked_data = data.where(combined_mask)
    regional_mean = masked_data.mean(dim=['lat', 'lon'])
    
    return regional_mean.astype(np.float64)

def detrend_data(x, y):
    """Remove linear trends from time series."""
    x_detrended = signal.detrend(x)
    y_detrended = signal.detrend(y)
    return x_detrended, y_detrended

def compute_partial_correlation(x, y, z):
    """Compute partial correlation between x and y, controlling for z."""
    # Remove z's influence from x
    z_reshaped = z.reshape(-1, 1)
    x_resid = x - OLS(x, add_constant(z_reshaped)).fit().predict(add_constant(z_reshaped))
    
    # Remove z's influence from y
    y_resid = y - OLS(y, add_constant(z_reshaped)).fit().predict(add_constant(z_reshaped))
    
    # Correlation of residuals
    return pearsonr(x_resid, y_resid)

print("Helper functions defined!")

In [ ]:
# Load data
print("Loading weekly climate data...")
ds_night, ds_day, ds_vpd, week_dates = load_weekly_data()

# Load state mask
ds_mask = xr.open_dataset(MASK_FILE)
state_mask = ds_mask.state_mask.load()
ds_mask.close()

# Load cattle data
df_cattle = pd.read_csv(CATTLE_DATA_FILE, parse_dates=['date'])

# Filter to overlapping period
df_cattle = df_cattle[
    (df_cattle['date'] >= pd.to_datetime(week_dates[0])) &
    (df_cattle['date'] <= pd.to_datetime(week_dates[-1]))
].copy()

print(f"Climate data: {len(week_dates)} weeks")
print(f"Cattle data: {len(df_cattle)} weeks")
print(f"Date range: {pd.to_datetime(week_dates[0]).date()} to {pd.to_datetime(week_dates[-1]).date()}")

## 1. Data Preparation and Integration

Aggregate VPD and temperature metrics by region and merge with cattle slaughter data.

In [ ]:
# Compute regional climate metrics
region_4_ids = CATTLE_REGION_IDS['region_4']
region_6_ids = CATTLE_REGION_IDS['region_6']
combined_ids = region_4_ids + region_6_ids

# VPD metrics
vpd_mean_r4 = compute_regional_mean(ds_vpd['vpd_mean'], region_4_ids, state_mask)
vpd_mean_r6 = compute_regional_mean(ds_vpd['vpd_mean'], region_6_ids, state_mask)
vpd_mean_combined = compute_regional_mean(ds_vpd['vpd_mean'], combined_ids, state_mask)

vpd_max_r4 = compute_regional_mean(ds_vpd['vpd_max'], region_4_ids, state_mask)
vpd_max_r6 = compute_regional_mean(ds_vpd['vpd_max'], region_6_ids, state_mask)
vpd_max_combined = compute_regional_mean(ds_vpd['vpd_max'], combined_ids, state_mask)

# Temperature metrics (for control analysis)
temp_day_r4 = compute_regional_mean(ds_day['hours_above_30'], region_4_ids, state_mask)
temp_day_r6 = compute_regional_mean(ds_day['hours_above_30'], region_6_ids, state_mask)
temp_day_combined = compute_regional_mean(ds_day['hours_above_30'], combined_ids, state_mask)

temp_night_r4 = compute_regional_mean(ds_night['hours_above_24'], region_4_ids, state_mask)
temp_night_r6 = compute_regional_mean(ds_night['hours_above_24'], region_6_ids, state_mask)
temp_night_combined = compute_regional_mean(ds_night['hours_above_24'], combined_ids, state_mask)

# Create integrated DataFrame
df_integrated = pd.DataFrame({
    'date': pd.to_datetime(week_dates),
    'vpd_mean_r4': vpd_mean_r4.values,
    'vpd_mean_r6': vpd_mean_r6.values,
    'vpd_mean_combined': vpd_mean_combined.values,
    'vpd_max_r4': vpd_max_r4.values,
    'vpd_max_r6': vpd_max_r6.values,
    'vpd_max_combined': vpd_max_combined.values,
    'temp_day_r4': temp_day_r4.values,
    'temp_day_r6': temp_day_r6.values,
    'temp_day_combined': temp_day_combined.values,
    'temp_night_r4': temp_night_r4.values,
    'temp_night_r6': temp_night_r6.values,
    'temp_night_combined': temp_night_combined.values,
})

# Merge with cattle data
df_merged = pd.merge(df_cattle, df_integrated, on='date', how='inner')

# Add temporal variables
df_merged['year'] = df_merged['date'].dt.year
df_merged['month'] = df_merged['date'].dt.month
df_merged['season'] = df_merged['month'].apply(
    lambda m: 'Winter' if m in [12, 1, 2] else 
              'Spring' if m in [3, 4, 5] else 
              'Summer' if m in [6, 7, 8] else 'Fall'
)

print(f"Integrated dataset: {df_merged.shape}")
print(f"\nSummary statistics:")
print(df_merged[['vpd_mean_combined', 'vpd_max_combined', 'region_4_beef_dairy', 'region_6_beef_dairy']].describe())

df_merged.head()

## 2. Exploratory Correlation Analysis

Initial assessment of VPD-mortality relationships using scatter plots and basic correlations.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# Define analysis pairs
analyses = [
    ('vpd_mean_combined', 'region_4_beef_dairy', 'Region 4: Mean VPD vs Slaughter'),
    ('vpd_mean_combined', 'region_6_beef_dairy', 'Region 6: Mean VPD vs Slaughter'),
    ('vpd_mean_combined', 'regions_4_6_total', 'Combined: Mean VPD vs Total Slaughter'),
    ('vpd_max_combined', 'region_4_beef_dairy', 'Region 4: Max VPD vs Slaughter'),
    ('vpd_max_combined', 'region_6_beef_dairy', 'Region 6: Max VPD vs Slaughter'),
    ('vpd_max_combined', 'regions_4_6_total', 'Combined: Max VPD vs Total Slaughter'),
]

# Add total column
df_merged['regions_4_6_total'] = df_merged['region_4_beef_dairy'] + df_merged['region_6_beef_dairy']

for idx, (vpd_var, cattle_var, title) in enumerate(analyses):
    ax = axes.flatten()[idx]
    
    # Remove NaN values
    mask = ~(np.isnan(df_merged[vpd_var]) | np.isnan(df_merged[cattle_var]))
    x = df_merged.loc[mask, vpd_var].values
    y = df_merged.loc[mask, cattle_var].values
    months = df_merged.loc[mask, 'month'].values
    
    # Scatter plot colored by month
    scatter = ax.scatter(x, y, c=months, cmap='twilight', alpha=0.5, s=20)
    
    # Regression line
    z = np.polyfit(x, y, 1)
    p = np.poly1d(z)
    x_line = np.linspace(x.min(), x.max(), 100)
    ax.plot(x_line, p(x_line), 'r--', linewidth=2, alpha=0.8)
    
    # Statistics
    r_pearson, p_pearson = pearsonr(x, y)
    r_spearman, p_spearman = spearmanr(x, y)
    
    stats_text = f'Pearson r = {r_pearson:.3f} (p={p_pearson:.3e})\n'
    stats_text += f'Spearman ρ = {r_spearman:.3f} (p={p_spearman:.3e})\n'
    stats_text += f'n = {len(x)}'
    
    ax.text(0.05, 0.95, stats_text, transform=ax.transAxes, 
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8),
            fontsize=9)
    
    ax.set_xlabel(vpd_var.replace('_', ' ').title() + ' (kPa)', fontsize=10)
    ax.set_ylabel(cattle_var.replace('_', ' ').title() + ' (1000 head/week)', fontsize=10)
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.grid(True, alpha=0.3)
    
    # Add colorbar for month
    if idx == 2:
        cbar = plt.colorbar(scatter, ax=ax)
        cbar.set_label('Month', rotation=270, labelpad=20)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '01_vpd_cattle_scatter.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n=== Overall Correlation Summary ===")
print(f"Combined regions VPD-mortality correlation: r = {r_pearson:.3f}")

## 3. Temporal Lag Analysis

Examine if VPD exposure in previous weeks predicts current mortality (cumulative stress effects).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

max_lag = 8  # weeks
analyses_lag = [
    ('vpd_mean_r4', 'region_4_beef_dairy', 'Region 4: Mean VPD'),
    ('vpd_mean_r6', 'region_6_beef_dairy', 'Region 6: Mean VPD'),
    ('vpd_max_r4', 'region_4_beef_dairy', 'Region 4: Max VPD'),
    ('vpd_max_r6', 'region_6_beef_dairy', 'Region 6: Max VPD'),
]

for idx, (vpd_var, cattle_var, title) in enumerate(analyses_lag):
    ax = axes[idx]
    
    lag_corrs = []
    lag_pvals = []
    
    for lag in range(0, max_lag + 1):
        # Create lagged VPD
        df_merged[f'{vpd_var}_lag{lag}'] = df_merged[vpd_var].shift(lag)
        
        # Compute correlation
        mask = ~(np.isnan(df_merged[f'{vpd_var}_lag{lag}']) | np.isnan(df_merged[cattle_var]))
        if mask.sum() > 10:
            r, p = pearsonr(
                df_merged.loc[mask, f'{vpd_var}_lag{lag}'],
                df_merged.loc[mask, cattle_var]
            )
            lag_corrs.append(r)
            lag_pvals.append(p)
        else:
            lag_corrs.append(np.nan)
            lag_pvals.append(np.nan)
    
    # Plot correlation vs lag
    lags = np.arange(0, max_lag + 1)
    bars = ax.bar(lags, lag_corrs, color='steelblue', alpha=0.7, edgecolor='black')
    
    # Color significant bars
    for i, (bar, p) in enumerate(zip(bars, lag_pvals)):
        if p < 0.05:
            bar.set_color('darkred')
            bar.set_alpha(0.8)
    
    ax.axhline(0, color='black', linewidth=1)
    ax.set_xlabel('Lag (weeks)', fontsize=11)
    ax.set_ylabel('Pearson Correlation', fontsize=11)
    ax.set_title(f'{title}\nLagged Correlation with Slaughter', fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    # Annotate values and significance
    for i, (lag, r, p) in enumerate(zip(lags, lag_corrs, lag_pvals)):
        if not np.isnan(r):
            label = f'{r:.2f}'
            if p < 0.001:
                label += '***'
            elif p < 0.01:
                label += '**'
            elif p < 0.05:
                label += '*'
            ax.text(lag, r + 0.005*np.sign(r), label, ha='center', 
                   va='bottom' if r > 0 else 'top', fontsize=8)
    
    # Find max correlation
    max_idx = np.nanargmax(np.abs(lag_corrs))
    ax.text(0.98, 0.02, f'Peak |r| at lag {max_idx}: {lag_corrs[max_idx]:.3f}',
           transform=ax.transAxes, ha='right', va='bottom',
           bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))

# Add legend for significance
fig.text(0.5, 0.02, '*p<0.05  **p<0.01  ***p<0.001  (red bars = significant)', 
         ha='center', fontsize=10, style='italic')

plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig(OUTPUT_DIR / '02_lag_correlation_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n=== Lag Analysis Summary ===")
for title, lag_corrs in zip([a[2] for a in analyses_lag], 
                            [lag_corrs for _ in analyses_lag]):
    max_idx = np.nanargmax(np.abs(lag_corrs))
    print(f"{title}: Peak at lag {max_idx} weeks (r = {lag_corrs[max_idx]:.3f})")

## 4. Seasonal Variation in VPD-Mortality Relationships

Assess whether VPD effects differ by season.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

seasons = ['Winter', 'Spring', 'Summer', 'Fall']
colors_season = ['blue', 'green', 'red', 'orange']

for idx, season in enumerate(seasons):
    ax = axes[idx]
    
    # Filter by season
    df_season = df_merged[df_merged['season'] == season].copy()
    
    if len(df_season) < 10:
        continue
    
    # Scatter plot
    mask = ~(np.isnan(df_season['vpd_mean_combined']) | np.isnan(df_season['regions_4_6_total']))
    x = df_season.loc[mask, 'vpd_mean_combined'].values
    y = df_season.loc[mask, 'regions_4_6_total'].values
    
    ax.scatter(x, y, alpha=0.5, s=30, color=colors_season[idx])
    
    # Regression
    if len(x) > 10:
        z = np.polyfit(x, y, 1)
        p = np.poly1d(z)
        x_line = np.linspace(x.min(), x.max(), 100)
        ax.plot(x_line, p(x_line), 'k--', linewidth=2, alpha=0.8)
        
        # Statistics
        r_pearson, p_pearson = pearsonr(x, y)
        r_spearman, p_spearman = spearmanr(x, y)
        
        stats_text = f'Pearson r = {r_pearson:.3f}\n'
        stats_text += f'p-value = {p_pearson:.3e}\n'
        stats_text += f'Spearman ρ = {r_spearman:.3f}\n'
        stats_text += f'n = {len(x)}'
        
        ax.text(0.05, 0.95, stats_text, transform=ax.transAxes,
               verticalalignment='top', bbox=dict(boxstyle='round', 
               facecolor=colors_season[idx], alpha=0.3), fontsize=9)
    
    ax.set_xlabel('Mean VPD (kPa)', fontsize=11)
    ax.set_ylabel('Total Slaughter (1000 head/week)', fontsize=11)
    ax.set_title(f'{season}\nVPD-Mortality Relationship', fontweight='bold', fontsize=12)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '03_seasonal_vpd_mortality.png', dpi=300, bbox_inches='tight')
plt.show()

# Summary table
print("\n=== Seasonal Correlation Summary ===")
print(f"{'Season':<10} {'Pearson r':<12} {'p-value':<12} {'n':<8}")
print("-" * 45)
for season in seasons:
    df_season = df_merged[df_merged['season'] == season]
    mask = ~(np.isnan(df_season['vpd_mean_combined']) | np.isnan(df_season['regions_4_6_total']))
    if mask.sum() > 10:
        x = df_season.loc[mask, 'vpd_mean_combined'].values
        y = df_season.loc[mask, 'regions_4_6_total'].values
        r, p = pearsonr(x, y)
        print(f"{season:<10} {r:<12.3f} {p:<12.3e} {len(x):<8}")

## 5. Partial Correlation: VPD Effects Controlling for Temperature

Determine if VPD has independent effects beyond temperature alone.

In [ ]:
# Compute partial correlations
results_partial = []

regions = [
    ('Region 4', 'vpd_mean_r4', 'temp_day_r4', 'region_4_beef_dairy'),
    ('Region 6', 'vpd_mean_r6', 'temp_day_r6', 'region_6_beef_dairy'),
    ('Combined', 'vpd_mean_combined', 'temp_day_combined', 'regions_4_6_total'),
]

for region_name, vpd_var, temp_var, cattle_var in regions:
    # Remove NaN
    mask = ~(np.isnan(df_merged[vpd_var]) | 
             np.isnan(df_merged[temp_var]) | 
             np.isnan(df_merged[cattle_var]))
    
    vpd = df_merged.loc[mask, vpd_var].values
    temp = df_merged.loc[mask, temp_var].values
    cattle = df_merged.loc[mask, cattle_var].values
    
    # Simple correlations
    r_vpd_cattle, p_vpd_cattle = pearsonr(vpd, cattle)
    r_temp_cattle, p_temp_cattle = pearsonr(temp, cattle)
    r_vpd_temp, p_vpd_temp = pearsonr(vpd, temp)
    
    # Partial correlations
    r_vpd_partial, p_vpd_partial = compute_partial_correlation(vpd, cattle, temp)
    r_temp_partial, p_temp_partial = compute_partial_correlation(temp, cattle, vpd)
    
    results_partial.append({
        'Region': region_name,
        'VPD-Cattle (r)': r_vpd_cattle,
        'VPD-Cattle (p)': p_vpd_cattle,
        'Temp-Cattle (r)': r_temp_cattle,
        'Temp-Cattle (p)': p_temp_cattle,
        'VPD-Temp (r)': r_vpd_temp,
        'VPD|Temp (r)': r_vpd_partial,
        'VPD|Temp (p)': p_vpd_partial,
        'Temp|VPD (r)': r_temp_partial,
        'Temp|VPD (p)': p_temp_partial,
        'n': len(vpd)
    })

df_partial = pd.DataFrame(results_partial)

# Visualization
fig, ax = plt.subplots(figsize=(14, 8))

x = np.arange(len(regions))
width = 0.15

# Plot bars
bars1 = ax.bar(x - width*2, df_partial['VPD-Cattle (r)'], width, 
               label='VPD-Cattle (zero-order)', color='darkblue', alpha=0.7)
bars2 = ax.bar(x - width, df_partial['VPD|Temp (r)'], width,
               label='VPD|Temp (partial)', color='lightblue', alpha=0.7)
bars3 = ax.bar(x + width, df_partial['Temp-Cattle (r)'], width,
               label='Temp-Cattle (zero-order)', color='darkred', alpha=0.7)
bars4 = ax.bar(x + width*2, df_partial['Temp|VPD (r)'], width,
               label='Temp|VPD (partial)', color='lightcoral', alpha=0.7)

ax.set_xlabel('Region', fontsize=12)
ax.set_ylabel('Correlation Coefficient', fontsize=12)
ax.set_title('Partial Correlation Analysis: Independent Effects of VPD and Temperature\non Cattle Mortality',
            fontweight='bold', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(df_partial['Region'])
ax.legend(loc='upper right', fontsize=10)
ax.axhline(0, color='black', linewidth=1)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bars in [bars1, bars2, bars3, bars4]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{height:.3f}', ha='center', va='bottom' if height > 0 else 'top',
               fontsize=8)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '04_partial_correlation_vpd_temp.png', dpi=300, bbox_inches='tight')
plt.show()

# Print results
print("\n=== Partial Correlation Results ===")
print(df_partial.to_string(index=False))

print("\n=== Key Findings ===")
for _, row in df_partial.iterrows():
    vpd_change = row['VPD-Cattle (r)'] - row['VPD|Temp (r)']
    temp_change = row['Temp-Cattle (r)'] - row['Temp|VPD (r)']
    print(f"\n{row['Region']}:")
    print(f"  VPD effect reduced by {abs(vpd_change):.3f} when controlling for temperature")
    print(f"  Temp effect reduced by {abs(temp_change):.3f} when controlling for VPD")
    if row['VPD|Temp (p)'] < 0.05:
        print(f"  VPD has significant INDEPENDENT effect (p={row['VPD|Temp (p)']:.3e})")
    else:
        print(f"  VPD effect not significant after controlling for temperature")